---
title: Linked-Read Barcode Summary
subtitle: A look at valid/invalid barcodes
date: "2026-07-08"
edit_url: null
---
**Linked-read type**: haplotagging

In [1]:
import altair as alt
from harpy.report.html import StatsBox
from harpy.report.plots import SafeRender
from harpy.report.tables import ITable
from harpy.report.utilities import trunc_digits
from harpy.report.theme import palette
import pandas as pd
from pathlib import Path


This report details the counts of valid and invalid linked-read barcodes in the 
sample read headers. Validation varies between linked-read technologies, look through the tabs in Supporting Info below
for more information on the different technologies. More exhaustive explanations can be found in 
[Harpy's documentation](https://pdimens.github.io/harpy/getting_started/linked_read_data/#linked-read-data-types).

In [2]:
indir = "reports/data"


In [3]:
# Parameters
indir = "/local/workdir/pd348/tripsacum/fastq/qc/reports/data"


In [4]:
infiles = list(Path(indir).glob("*.bxcount"))

bx_stats_list = []
invalids_list = []

for infile in infiles:
    samplename = infile.stem.replace(".count", "")

    # Read data
    s_df = pd.read_table(infile, header=None, names=['V1', 'V2'])

    # Extract values
    total = s_df.loc[0, 'V2']
    bc_total = s_df.loc[1, 'V2']
    bc_ok = s_df.loc[2, 'V2']
    bc_invalid = s_df.loc[3, 'V2']

    # Calculate percentages
    if bc_total == 0:
        bc_percent = 0
        ig_percent = 0
    else:
        bc_percent = trunc_digits((bc_ok / bc_total), 4)
        ig_percent = trunc_digits((bc_invalid / bc_total), 4)

    # Append to BXstats
    bx_stats_list.append({
        'Sample': samplename,
        'TotalReads': total,
        'TotalBarcodes': bc_total,
        'ValidBarcodes': bc_ok,
        'ValidProportion': bc_percent,
        'InvalidBarcodes': bc_invalid,
        'InvalidProportion': ig_percent
    })

    # Build invalids data
    invalid_data = {'Sample': samplename, 'InvalidBarcodes': bc_invalid}
    if len(s_df) > 4:
        for idx in range(4, len(s_df)):
            invalid_data[s_df.loc[idx, 'V1']] = s_df.loc[idx, 'V2']

    invalids_list.append(invalid_data)

# Create final DataFrames
BXstats = pd.DataFrame(bx_stats_list)
invalids_percent = pd.DataFrame(invalids_list)


In [5]:
cols_to_convert = invalids_percent.columns[2:]
invalids_percent[cols_to_convert] = (
    invalids_percent[cols_to_convert].astype('float64')
    .div(invalids_percent['InvalidBarcodes'], axis=0)
    .round(3)
)

# Merge BXstats (first 5 columns) with invalids_percent (excluding 2nd column)
stats = pd.merge(
    BXstats.iloc[:, :5], 
    invalids_percent.drop(columns=invalids_percent.columns[1]),
    on='Sample'
)

invalids_percent = invalids_percent.drop(columns = ['Sample','InvalidBarcodes'])

# Pivot to long format
invalids_long = invalids_percent.melt(
    var_name='Position', 
    value_name='Invalid'
)


In [6]:
_bxmin = BXstats['ValidProportion'].min().round(4)
_bxmax = BXstats['ValidProportion'].max().round(4)
_bxmean = BXstats['ValidProportion'].mean().round(4)

(
    StatsBox()
    .add(len(BXstats.index), "Files")
    .conditional(_bxmin, "Min Valid", 0.3, as_percent=True)
    .conditional(_bxmax, "Max Valid", 0.75, as_percent=True)
    .conditional(_bxmean, "Mean Valid", 0.5, as_percent=True)
    .render()
)


## Barcode Statistics

Below is a table[^1] and its corresponding figure listing all the samples Harpy processed and their associated
barcode statistics as determined by the sequences in the **forward-read file (R1) only**.
If for some reason `TotalBarcodes` equals `0`, then there may be an issue with
the format of your FASTQ headers.

[^1]:
    | Column Name | Description |
    |:------------|:------------|
    | Sample  | name of the sample |
    | TotalReads  | total number of reads |
    |  TotalBarcodes | total number of barcodes present |
    |  ValidBarcodes | number of valid linked-read barcodes |
    |  ValidProportion | percent of linked-read barcodes that are valid for that chemistry |
    |  ... | percent invalidations at this barcode segment/position |


In [7]:
percdata = (
    BXstats[['Sample', 'ValidProportion', 'InvalidProportion']]
    .rename(columns={'ValidProportion': 'Valid', 'InvalidProportion': 'Invalid'})
)
del BXstats


In [8]:
#| label: eCvx29p8dep4zf4_validtable
ITable(stats, "fastq.bx.stats.csv", fixedcols=1).render()


In [9]:
#| label: eCvx29p8dep4zf4_validplot
with SafeRender(percdata):
    (
        alt.Chart(percdata)
        .transform_density(
            'Valid',
            as_=['Valid Barcodes', 'density'],
            extent=[0,1],
            bandwidth=0.05
        )
        .mark_area(opacity = 1, line=False, interpolate="basis")
        .encode(
            x=alt.X("Valid Barcodes:Q").axis(format='%'),
            y=alt.Y('density:Q').title("Density").axis(labels=False),
            tooltip = [
                alt.Tooltip("Valid Barcodes:Q", bin=True, title="Percent Valid", format = ".0%")
            ]
        )
        .properties(
            title='Percent Valid Barcodes Across Samples',
            usermeta={'embedOptions': {'downloadFileName': 'valid.barcodes.percent'}}
        )
    ).display()


alt.Chart(...)

## Invalid Positions in Barcodes
This is the density distribution of invalid barcode positions across all samples as a percent of all the invalid barcodes.
This should help you assess whether some positions have a higher incidence of failure than others, which may be diagnostic
of issues with the chemistry. You can isolate any of the positions by clicking on them in the figure legend.

In [10]:
selection = alt.selection_point(fields=['Position'], bind='legend')
with SafeRender(invalids_long):
    (
        alt.Chart(invalids_long)
        .transform_density(
            density='Invalid',
            groupby=['Position'],
            extent=[0, 1],
            bandwidth=0.05  # Adjust this to control smoothness
        )
        .mark_area(opacity = 1, line=False, interpolate="natural")
        .encode(
            x=alt.X('value:Q', title='Percent Invalid').axis(format="%"),
            y=alt.Y('density:Q', title='Density')
                .axis(labels=False)
                .stack(False),
            color= alt.Color('Position:N')
                .title("Position")
                .scale(range=palette(invalids_long['Position'].nunique())),
            opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
            tooltip = [
                "Position:N",
                alt.Tooltip("value:Q", bin=True, format = ".0%", title="Percent Invalid")
            ],
        )
    .add_params(selection)
    .properties(
            title='Invalid Barcode Positions Across Samples',
            usermeta={'embedOptions': {'downloadFileName': 'invalid.bc.positions'}}
        )
    ).display()


alt.Chart(...)

## Supporting Info

```{dropdown} Linked-Read Chemistries
::::{tab-set} 
:::{tab-item} Haplotagging
Segments: the `Axx`, `Bxx`, `Cxx`, `Dxx` parts of haplotagging barcodes

The correct haplotagging format is `BX:Z:AxxCxxBxxDxx`, where 
there is a tab before (and possibly after) `BX:Z` and `xx` are 2-digit numbers
between `00` and `96`. Additional tags (e.g. `QX:Z` or `VX:i`) are permissible as long as they conform to the 
[SAM comment spec (section 1.5)](https://samtools.github.io/hts-specs/SAMv1.pdf) 
of `TAG:TYPE:VALUE`.


![haplotagging FASTQ record](https://pdimens.github.io/harpy/static/lr_haplotagging.svg)
:::

:::{tab-item} stLFR
Segments: the `X`,`Y`,`Z` segments of stFLR barcodes  

The correct stLFR format is `#X_Y_Z` at the end of the sequence ID, where `X`, `Y`, and `Z` are
integers of any value, with `0` as the value for any of the segments being considered invalid (e.g. `32_0_1101`).

![stLFR FASTQ record](https://pdimens.github.io/harpy/static/lr_stlfr.svg)
:::

:::{tab-item} TELLseq
Segments: TELLseq is not combinatorial, but for the purposes of these assessments, the "segments"
are the nucleotide position in the barcode.

The correct TELLseq format is `:ATCGN` at the end of the sequence ID, where `ATCGN` are
uppercase nucleotides (`A`, `T`, `C`, `G`, `N`), with `N` being considered invalid. TELLseq
uses 18bp barcodes, so the barcodes are generally expected to be 18bp long and the presence
of `N` would make the barcode considered invalid.

![TELLseq FASTQ record](https://pdimens.github.io/harpy/static/lr_tellseq.svg)
:::
::::
```